In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("HF Token: ").strip()
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=True)
print("✅ Authenticated")

In [ ]:
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
!git clone {REPO_URL}
%cd AI_PROJECT_TAMER_Complete/tamer_ocr
!pip install -r requirements.txt
!pip install datasets huggingface_hub

In [ ]:
import os
import re
from huggingface_hub import HfApi, hf_hub_download

MODEL_REPO = "JJKK1212/tamer-math-ocr" 
os.makedirs('./checkpoints', exist_ok=True)

try:
    api = HfApi(token=os.environ['HF_TOKEN'])
    files = api.list_repo_files(repo_id=MODEL_REPO, repo_type="model")
    ckpt_files = [f for f in files if re.match(r"checkpoint_epoch_\d+\.pt", f)]
    if ckpt_files:
        latest = max(ckpt_files, key=lambda x: int(re.findall(r"\d+", x)[0]))
        path = hf_hub_download(repo_id=MODEL_REPO, filename=latest, local_dir="./checkpoints", repo_type="model", token=os.environ['HF_TOKEN'])
        if os.path.exists("./checkpoints/latest.pt"): os.remove("./checkpoints/latest.pt")
        os.rename(path, "./checkpoints/latest.pt")
        print("✅ Checkpoint Synced")
except Exception as e: print(f"ℹ️ Hub sync skipped: {e}")

In [ ]:
# PHASE 0: Global Language Pre-training
# Decoder trains on ALL LaTeX strings from ALL datasets (150k+ samples)
# This makes it a Master of LaTeX grammar across all domains.
!python train.py \
    --datasets im2latex-100k mathwriting crohme hme100k \
    --phase0 \
    --batch-size 128 \
    --num-epochs 30 \
    --lr 1e-3 \
    --num-workers 4 \
    --download \
    --output-dir ./checkpoints/phase0

In [ ]:
# PHASE 1: Printed OCR
!python train.py \
    --datasets im2latex-100k \
    --resume ./checkpoints/phase0/best.pt \
    --batch-size 64 \
    --freeze-epochs 5 \
    --num-epochs 40 \
    --lr 3e-4 \
    --num-workers 4 \
    --output-dir ./checkpoints/phase1

In [ ]:
# PHASE 2: Clean Handwriting
!python train.py \
    --datasets mathwriting \
    --replay-datasets im2latex-100k \
    --resume ./checkpoints/phase1/best.pt \
    --batch-size 64 \
    --enc-lr-mult 1.0 \
    --dec-lr-mult 0.1 \
    --num-epochs 40 \
    --lr 1e-4 \
    --num-workers 4 \
    --output-dir ./checkpoints/phase2

In [ ]:
# PHASE 3: Competition Handwriting
!python train.py \
    --datasets crohme \
    --replay-datasets mathwriting \
    --resume ./checkpoints/phase2/best.pt \
    --batch-size 64 \
    --enc-lr-mult 1.0 \
    --dec-lr-mult 0.1 \
    --num-epochs 50 \
    --lr 5e-5 \
    --num-workers 4 \
    --output-dir ./checkpoints/phase3

In [ ]:
# PHASE 4: Real-World Camera Photos
!python train.py \
    --datasets hme100k \
    --replay-datasets crohme \
    --resume ./checkpoints/phase3/best.pt \
    --batch-size 64 \
    --enc-lr-mult 1.0 \
    --dec-lr-mult 0.05 \
    --num-epochs 80 \
    --lr 1e-5 \
    --num-workers 4 \
    --output-dir ./checkpoints/phase4